# TP 7 – Ce client laissera-t-il un pourboire ? (50 min)

**Objectif** : classer les courses payées **par carte** (les pourboires en espèces ne sont pas enregistrés) :
pourboire ou pas. Comparer Random Forest et SVM linéaire, tracer la courbe ROC.

**Réussi si** : AUC > 0,65 et courbe ROC tracée.

In [ ]:
# Se placer à la racine du dépôt : les chemins data/... sont relatifs à la racine
import os
while not os.path.isdir("data") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
print("Répertoire de travail :", os.getcwd())

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
spark = SparkSession.builder.appName("TP").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
zones = spark.read.csv("data/taxi_zone_lookup.csv", header=True, inferSchema=True)

from pyspark.ml.classification import RandomForestClassifier, LinearSVC
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
import matplotlib.pyplot as plt

### 1. Les données

- lire `data/trips.parquet` ; `heure` = `F.hour("tpep_pickup_datetime")`
- joindre les zones pour obtenir `Borough` : `zones.select(F.col("LocationID").alias("PULocationID"), "Borough")`
- garder `trip_distance > 0` **et** `payment_type = 1` (carte)
- `label` = 1.0 si `tip_amount > 0`, sinon 0.0 (`.cast("double")`)
- features : `trip_distance`, `heure`, `passenger_count`, `manhattan` (Borough == "Manhattan"), `nuit` (heure ≥ 20 ou < 5)

Quel est le taux de pourboire ? Les classes sont-elles équilibrées ?

In [ ]:
df = ...
print(df.agg(F.avg("label")).first()[0])
train, test = df.randomSplit([0.8, 0.2], seed=7)

### 2. Pipeline : `VectorAssembler` → `StandardScaler` → `RandomForestClassifier(numTrees=50, maxDepth=6)`, puis la même chose avec `LinearSVC`

In [ ]:
...

### 3. AUC des deux modèles

Indice : `BinaryClassificationEvaluator(metricName="areaUnderROC")`.

Si LinearSVC affiche `ERROR OWLQN: Failure! Resetting history` : c'est un message interne de l'optimiseur, sans conséquence.

In [ ]:
...

### 4. Matrice de confusion (Random Forest, seuil 0,5)

Indice : `pred.groupBy("label").pivot("prediction").count()`. Calculez précision et rappel à la main.

In [ ]:
...

### 5. Courbe ROC en Matplotlib

Récupérer `label` et la probabilité de la classe 1 : `vector_to_array("probability")[1]`, puis `toPandas()` (le test est petit).
Les probabilités sont concentrées entre 0,3 et 0,7 : prenez comme seuils les **quantiles** des scores (`numpy.quantile(p, numpy.linspace(0, 1, 50))`) plutôt que 20 seuils réguliers.
Pour chaque seuil, calculer le taux de vrais positifs et le taux de faux positifs, puis `plt.plot(fpr, tpr)`.

In [ ]:
scores = ...
...

### 6. Sauvegarder le pipeline **Random Forest** dans `out/modele_pourboire`

`rf_model.write().overwrite().save("out/modele_pourboire")`

Pourquoi le Random Forest et pas LinearSVC, alors que leurs AUC sont presque égales ? Le TP 11 a besoin d'une **probabilité** (colonne `probability`), que LinearSVC ne fournit pas.

In [ ]:
...

### Bonus – `CrossValidator` + `ParamGridBuilder` sur `numTrees` et `maxDepth` (3 plis). C'est plus long !

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
...

### Bonus 2 – Les vraies données

`python scripts/download_official.py` puis relancez ce notebook. Les résultats sont-ils aussi nets ?